In [ ]:
import os
import random
from tqdm import tqdm
import json
input_folder = "/home/junzhewu/data/isaac_scenes_v1/grscenes_commercial_test/models/"

def find_object_folder(prim_path):
    # extract model_name from prim_path
    model_name = prim_path.split("model_")[-1].split("_")[0]
    # iterate over the input_folder to find a folder that contains the prim_path
    for root, dirs, files in os.walk(input_folder):
        for dir in dirs:
            if model_name in dir:
                return f"{root}/{dir}"
                break
    return None

In [ ]:
find_object_folder("/Root/Meshes/Furnitures/person/model_3c0959833a5106fabcc1c95e8f0d2280_0/Instance/SM_6IQDDURVAV25EPTUKQ888888/component1")

In [ ]:
# create info.json for each object
all_files = []
for root, dirs, files in os.walk(input_folder):
    if "instance_renamed.usd" in files:
        all_files.append((root, files))
random.shuffle(all_files)
pbar = tqdm(all_files, desc="Processing info.json files", maxinterval=0.5)
for root, files in pbar:
    info = {
        "decimated_face_count": 0,
        "decimation_success": "instance_renamed_decimated.usd" in files,
    }
    with open(os.path.join(root, "info.json"), "w") as f:
        json.dump(info, f)


In [ ]:
# check problematic objects, sort and keep unique ones
objects = {}
with open("problem_objects.log", "r") as f:
    for line in f:
        usd_path, face_count = line.strip().split(",")
        if usd_path in objects:
            continue
        else:
            objects[usd_path] = int(face_count)

objects = sorted(objects.items(), key=lambda x: x[1], reverse=True)
for i, (usd_path, face_count) in enumerate(objects):
    print(f"{i+1}. {usd_path}, {face_count}")

with open("problem_objects.log", "w") as f:
    for i, (usd_path, face_count) in enumerate(objects):
        f.write(f"{usd_path}, {face_count}\n")

In [ ]:
# remove problematic objects
with open("problem_objects.log", "w") as f:
    for i, (usd_path, face_count) in enumerate(objects):
        f.write(f"{usd_path}, {face_count}\n")
        # remove instance_renamed_decimated.usd
        decimated_usd_path = usd_path.replace("instance_renamed.usd", "instance_renamed_decimated.usd")
        if os.path.exists(decimated_usd_path):
            os.remove(decimated_usd_path)
            print(f"Removed {decimated_usd_path}")

In [ ]:
# remove heavy objects
with open("heavy_objects.csv", "r") as f:
    lines = f.readlines()
    for line in lines[1:]:
        if "," in line:
            scene_path, prim_path, prim_name, face_count, source_usd_folder = line.strip().split(",")
            source_usd_path = find_object_folder(prim_path) + "/instance_renamed_decimated.usd"
            if int(face_count)>30000:
                print(face_count, source_usd_path)
                if os.path.exists(source_usd_path):
                    os.remove(source_usd_path)
                    print(f"Removed {source_usd_path}")
                else:
                    print(f"File not found: {prim_path}")


In [ ]:
prim_path = "/Root.001/Meshes.001/Furnitures.001/other.004/model_c9d109f5274e8b9304598fed6ed3ea20_2"
# extract model_name from prim_path
model_name = prim_path.split("model_")[-1].split("_")[0]
# iterate over the input_folder to find a folder that contains the prim_path
for root, dirs, files in os.walk(input_folder):
    for dir in dirs:
        if model_name in dir:
            print(f"{root}/{dir}")
            break


In [ ]:
# remove person mesh
for root, dirs, files in os.walk(input_folder):
    if "person" in root and "instance_renamed_decimated.usd" in files:
        os.remove(os.path.join(root, "instance_renamed_decimated.usd"))
        print(f"Removed {os.path.join(root, 'instance_renamed_decimated.usd')}")

In [ ]:
# remove blender files
input_folder = "/home/junzhewu/data/isaac_scenes_v1/grscenes_commercial_test/models/"
for root, dirs, files in os.walk(input_folder):
    # file_name = "instance_renamed_decimated.blend"
    # file_name = "instance_renamed_decimated.usd"
    file_name = "info.json"
    file_name = "skipped.txt"
    if file_name in files:
        os.remove(os.path.join(root, file_name))
        print(f"Removed {os.path.join(root, file_name)}")

In [ ]:
# remove blender files according to the info.json
input_folder = "/home/junzhewu/data/isaac_scenes_v1/grscenes_commercial_test/models/"
for root, dirs, files in os.walk(input_folder):
    # file_name = "instance_renamed_decimated.blend"
    # file_name = "instance_renamed_decimated.usd"
    file_name = "info.json"
    if file_name in files:
        with open(os.path.join(root, file_name), "r") as f:
            info = json.load(f)
            if info["decimated_face_count"]==0:
                print(f"Removed {os.path.join(root, 'instance_renamed_decimated.usd')}")
                os.remove(os.path.join(root, "instance_renamed_decimated.usd"))

In [ ]:
# restore instance.usd
from tqdm import tqdm
import shutil
input_folder = "/home/junzhewu/data/isaac_scenes_v1/grscenes_commercial_test/models/"
folders = []
def remove_if_exists(path):
    if os.path.exists(path):
        os.remove(path)
for root, dirs, files in os.walk(input_folder):
    folders.append([root, dirs, files])
for root, dirs, files in tqdm(folders):
    if "info.json" in files and not "instance_renamed.usd" in files:
        raw_folder = root.replace("grscenes_commercial_test", "grscenes_commercial_raw")
        print(f"Restoring {root} to {raw_folder}")
        remove_if_exists(os.path.join(root, "info.json"))
        remove_if_exists(os.path.join(root, "optimized.txt"))
        remove_if_exists(os.path.join(root, "instance_renamed_decimated.usd"))
        remove_if_exists(os.path.join(root, "instance_renamed.usd"))
        remove_if_exists(os.path.join(root, "instance.usd"))
        remove_if_exists(os.path.join(root, "skipped.txt"))
        # copy file
        shutil.copy(os.path.join(raw_folder, "instance.usd"), os.path.join(root, "instance.usd"))

In [ ]:
import re
# check optimized models
input_folder = "/home/junzhewu/data/isaac_scenes_v1/grscenes_commercial_test/models/"
for root, dirs, files in os.walk(input_folder):
    file_name = "optimized.txt"
    if file_name in files:
        with open(os.path.join(root, file_name), "r") as f:
            optimized_face_count = int(re.findall(r"Total triangles in optimized meshes: (\d+)", f.read())[0])
            if optimized_face_count == 0:
                os.remove(os.path.join(root, "instance_renamed_decimated.usd"))
                print(f"Removed {os.path.join(root, 'instance_renamed_decimated.usd')}")

In [ ]:
input_folder = "/home/junzhewu/data/isaac_scenes_v1/grscenes_commercial_test/models/"
material_folder = "../../../../../Materials"
for root, dirs, files in os.walk(input_folder):
    if "Materials" in dirs:
        # check if it is a folder or a link
        if not os.path.islink(os.path.join(root, "Materials")):
            print(f"Folder found: {os.path.join(root, 'Materials')}")
            # remove the folder
            shutil.rmtree(os.path.join(root, "Materials"))
            print(f"Removed {os.path.join(root, 'Materials')}")
            # create a link to the material folder
            os.symlink(material_folder, os.path.join(root, "Materials"))
            print(f"Linked {os.path.join(root, 'Materials')} to {material_folder}")
            # list os.path.join(root, "Materials")
            # print(os.listdir(os.path.join(root, material_folder)))
